In [1]:
import os

REPO = 'giomamaca/walmart-sales-forecasting'

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import userdata

    github_token = userdata.get('GITHUB_TOKEN')
    repo_dir = f'/content/{REPO.split("/")[1]}'
    if not os.path.exists(repo_dir):
        !git clone -q https://{github_token}@github.com/{REPO}.git {repo_dir}
    os.chdir(repo_dir)
    !git pull -q

    !pip install -q mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 90.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 90.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.0/121.0 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.

In [2]:
import mlflow

!mlflow db upgrade sqlite:///mlflow.db
mlflow.set_tracking_uri('sqlite:///mlflow.db')

2026/07/11 18:05:44 INFO mlflow.store.db.utils: Updating database tables


In [3]:
ARCHS = sorted(e.name[:-len('_Training')] for e in mlflow.MlflowClient().search_experiments()
              if e.name.endswith('_Training'))

client = mlflow.MlflowClient()

# diagnostikam achvena rom kaggle-is qcevas yvelaze ukot test-formis holdout-i
# (wmae_holdout) imeorebs da ara CV. amitom arqiteqturebi romlebsac es metrika
# aqvt am metrikit dardeba ertmanets; dzvel modelebs (holdout-is gareshe)
# mxolod mashin vadarebt CV-it (fold 1-2), tu holdout arc ert models ar aqvs
def arch_metrics(arch):
    exp = client.get_experiment_by_name(f'{arch}_Training')
    if exp is None:
        return None
    runs = client.search_runs(exp.experiment_id, filter_string=f"attributes.run_name = '{arch}_CV'",
                              order_by=['attributes.start_time DESC'], max_results=1)
    if not runs:
        return None
    m = runs[0].data.metrics
    folds = client.get_metric_history(runs[0].info.run_id, 'wmae')
    mature = [x.value for x in folds if x.step >= 1]
    cv = sum(mature) / len(mature) if mature else m.get('wmae_mean')
    return {'holdout': m.get('wmae_holdout'), 'cv': cv}

scores = {arch: metrics for arch in ARCHS if (metrics := arch_metrics(arch)) is not None}
with_holdout = {a: s['holdout'] for a, s in scores.items() if s['holdout'] is not None}
if with_holdout:
    best_arch = min(with_holdout, key=with_holdout.get)
else:
    best_arch = min(scores, key=lambda a: scores[a]['cv'])
scores, best_arch

({'DLinear': {'holdout': None, 'cv': 2631.2060856387434},
  'DLinear_v2': {'holdout': None, 'cv': 2616.5903098494273},
  'LightGBM': {'holdout': None, 'cv': 2555.506397760865},
  'LightGBM_v2': {'holdout': None, 'cv': 2409.233599909883},
  'LightGBM_v3': {'holdout': None, 'cv': 2031.4473339773474},
  'LightGBM_v4': {'holdout': 2238.581955162577, 'cv': 2002.6212149442804},
  'LightGBM_v5': {'holdout': 1935.7122369223732, 'cv': 1787.032340667387},
  'PatchTST': {'holdout': 3310.075007511755, 'cv': 2798.3787865197155},
  'Prophet': {'holdout': None, 'cv': 1858.7321831929285},
  'Prophet_v2': {'holdout': None, 'cv': 1858.522989091513},
  'TimesFM': {'holdout': 2753.9109653595133, 'cv': 2351.505565422921},
  'XGBoost': {'holdout': None, 'cv': 2753.128491216854},
  'XGBoost_v2': {'holdout': None, 'cv': 2365.098540060736}},
 'LightGBM_v5')

In [4]:
MODEL_NAME = 'walmart-sales-model'

def final_run(arch):
    exp = client.get_experiment_by_name(f'{arch}_Training')
    if exp is None:
        return None
    runs = client.search_runs(exp.experiment_id,
                              filter_string=f"attributes.run_name = '{arch}_Final'",
                              order_by=['attributes.start_time DESC'], max_results=1)
    return runs[0] if runs else None

def register(run, name):
    existing = [m.name for m in client.search_registered_models()]
    cur = client.get_registered_model(name).latest_versions[0] if name in existing else None
    if cur is None or cur.run_id != run.info.run_id:
        mlflow.register_model(f'runs:/{run.info.run_id}/model', name)

# yovel arqiteqturis sauketeso (Final) pipeline registrdeba tavisi saxelit,
# rom Model Registry-shi yvela arqiteqtura chans da chamoitvirtos saxelit
for arch in ARCHS:
    run = final_run(arch)
    if run is None:
        continue
    try:
        register(run, f'walmart-{arch}')
    except Exception as e:
        print('skip', arch, ':', e)

# saerto sauketeso modeli (holdout-it) icvleba kanonikuri saxelit walmart-sales-model
register(final_run(best_arch), MODEL_NAME)

[m.name for m in client.search_registered_models()]

Successfully registered model 'walmart-DLinear'.
2026/07/11 18:05:47 WARNING mlflow.tracking._model_registry.fluent: Run with id 3c179bd33460491bb8e8f591700c0326 has no artifacts at artifact path 'model', registering model based on models:/m-28cc591d849d4b439cd95feb3d70a484 instead
Created version '1' of model 'walmart-DLinear'.
Successfully registered model 'walmart-DLinear_v2'.
2026/07/11 18:05:47 WARNING mlflow.tracking._model_registry.fluent: Run with id cbaa1b5dc5b548699f39101bba8d1722 has no artifacts at artifact path 'model', registering model based on models:/m-d1484bf886cf4b0bbd8ba2fa37df7edf instead
Created version '1' of model 'walmart-DLinear_v2'.
Successfully registered model 'walmart-LightGBM'.
2026/07/11 18:05:47 WARNING mlflow.tracking._model_registry.fluent: Run with id cab1bc4bea1a4bcf8ab4658dc842f1c1 has no artifacts at artifact path 'model', registering model based on models:/m-1f2cbd75f35d44038f59b753127070e3 instead
Created version '1' of model 'walmart-LightGBM'.

['walmart-DLinear',
 'walmart-DLinear_v2',
 'walmart-LightGBM',
 'walmart-LightGBM_v2',
 'walmart-LightGBM_v3',
 'walmart-LightGBM_v4',
 'walmart-LightGBM_v5',
 'walmart-PatchTST',
 'walmart-Prophet',
 'walmart-Prophet_v2',
 'walmart-TimesFM',
 'walmart-XGBoost',
 'walmart-XGBoost_v2',
 'walmart-sales-model']

In [5]:
if IN_COLAB:
    import requests
    gh_user = requests.get('https://api.github.com/user', headers={'Authorization': f'token {github_token}'}).json()['login']
    !git config user.name "{gh_user}"
    !git config user.email "{gh_user}@users.noreply.github.com"

    !git add mlflow.db mlruns
    !git commit -m "Register best model"
    !git push

[main a91da27] Register best model
 1 file changed, 0 insertions(+), 0 deletions(-)
Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 2 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (3/3), 2.01 KiB | 294.00 KiB/s, done.
Total 3 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/giomamaca/walmart-sales-forecasting.git
   09beabe..a91da27  main -> main
